In [1]:
from openai import OpenAI
import numpy as np

client = OpenAI(base_url="http://127.0.0.1:1234/v1",api_key="lm-studio")
EMBED_MODEL = "text-embedding-bge-m3"
CHAT_MODEL = "qwen3.6-35b-a3b"

def embed(text):
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [2]:
with open("sample.txt", encoding="utf-8") as f:
    text = f.read()

print("文字数：", len(text))

文字数： 593


- `open()`:ファイルを開く。`encoding="utf-8`は日本語を正しく読み取るため
- `with`:使い終わったら自動でファイルを閉じてくれる

In [ ]:
def chunk_text(text, size=200, overlap=30): # 200文字ずつ切る。ただし、30文字ずつ重ねる
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = chunk_text(text)

for i, c in enumerate(chunks):
    print(f"---{i}番---")
    print(c)

---0番---
演習室利用ガイド（架空）

第1章 利用時間
演習室は平日9時から21時まで利用できます。土曜日は10時から17時まで開室しますが、日曜日と祝日は閉室です。定期試験の2週間前からは、平日の利用時間が22時まで延長されます。

第2章 入退室
入室には学生証のICカードが必要です。学生証を忘れた場合は、事務室で当日限りの仮カードを借りられます。仮カードは17時までに返却してください。

第3章 PC
---1番---
す。仮カードは17時までに返却してください。

第3章 PCの利用
演習室には40台のPCがあり、OSはUbuntuです。ログインには大学のアカウントを使います。パスワードを5回間違えると、30分間ログインできなくなります。個人のファイルはホームフォルダに保存し、デスクトップには置かないでください。デスクトップのファイルは毎晩自動で削除されます。

第4章 印刷
印刷は1人あたり月100枚まで無料
---2番---
れます。

第4章 印刷
印刷は1人あたり月100枚まで無料です。それを超える場合は1枚10円かかります。カラー印刷はできません。

第5章 禁止事項
演習室内での飲食は禁止です。ただし、ふた付きの飲み物は持ち込めます。PCへのソフトウェアのインストールは禁止されています。

第6章 トラブル
PCが動かなくなった場合は、電源を切らずに、TA（ティーチングアシスタント）に声をかけてください。TAは
---3番---
TA（ティーチングアシスタント）に声をかけてください。TAは平日の13時から18時まで常駐しています。それ以外の時間は、内線1234で情報センターに連絡してください。


In [9]:
vecs = [embed(c) for c in chunks] # chunksの中身を1個ずつ取り出し、関数embedを実行したものをリストにする
print(len(vecs), "個の塊を数字にした")

7 個の塊を数字にした


In [10]:
def search(question, k=2): # 質問を数字にして、全部の塊と似ていいる度を計算
    q_vec = embed(question) 
    scores = [cos_sim(q_vec, v) for v in vecs]
    ranked = sorted(zip(scores, chunks), reverse=True) # zipで「点数」と「塊」をペアにし、sorted(..., reverse=True)で点数の高い順に並べる
    return ranked[:k] # 上位2個だけ返す k=2なので

for score, chunk in search("TAは何時に居るの？"):
    print(f"{score:.3f}")
    print(chunk)
    print("--------")

0.622
第6章 トラブル
PCが動かなくなった場合は、電源を切らずに、TA（ティーチングアシスタント）に声をかけてください。TAは平日の13時から18時まで常駐しています。それ以外の時間は、内線1234で情報センターに連絡してください。
--------
0.474
第1章 利用時間
演習室は平日9時から21時まで利用できます。土曜日は10時から17時まで開室しますが、日曜日と祝日は閉室です。定期試験の2週間前からは、平日の利用時間が22時まで延長されます。
--------


# 問題点
- 指定した条件通りの答えが帰ってこない
- 答えが2つの塊に別れてしまって検索で片方しか拾えないとAIは正しく答えることができない

# overlapは何のためにある？
30文字重ねているのは切れ目の被害を減らすため。しかし根本的な解決にはならない。

In [8]:
### 修正 ###
def chunk_by_paragraph(text): # 200文字ずつ切る。ただし、30文字ずつ重ねる
    parts = text.split("\n\n")
    return [p.strip() for p in parts if p.strip()]

chunks = chunk_by_paragraph(text)

for i, c in enumerate(chunks):
    print(f"---{i}番---")
    print(c)

---0番---
演習室利用ガイド（架空）
---1番---
第1章 利用時間
演習室は平日9時から21時まで利用できます。土曜日は10時から17時まで開室しますが、日曜日と祝日は閉室です。定期試験の2週間前からは、平日の利用時間が22時まで延長されます。
---2番---
第2章 入退室
入室には学生証のICカードが必要です。学生証を忘れた場合は、事務室で当日限りの仮カードを借りられます。仮カードは17時までに返却してください。
---3番---
第3章 PCの利用
演習室には40台のPCがあり、OSはUbuntuです。ログインには大学のアカウントを使います。パスワードを5回間違えると、30分間ログインできなくなります。個人のファイルはホームフォルダに保存し、デスクトップには置かないでください。デスクトップのファイルは毎晩自動で削除されます。
---4番---
第4章 印刷
印刷は1人あたり月100枚まで無料です。それを超える場合は1枚10円かかります。カラー印刷はできません。
---5番---
第5章 禁止事項
演習室内での飲食は禁止です。ただし、ふた付きの飲み物は持ち込めます。PCへのソフトウェアのインストールは禁止されています。
---6番---
第6章 トラブル
PCが動かなくなった場合は、電源を切らずに、TA（ティーチングアシスタント）に声をかけてください。TAは平日の13時から18時まで常駐しています。それ以外の時間は、内線1234で情報センターに連絡してください。


In [ ]:
def ask(question):
    results = search(question, k=2)
    context = "\n---\n".join(chunk for score, chunk in results)

    messages = [
        {"role": "system", "content": "資料だけを根拠に答えて。資料にないことは「資料に記載にありません」と答えて。"},
        {"role": "user", "content": f"資料:\n{context}\n\n質問: {question}"},
    ]

    response = client.chat.completions.create(
        model=CHAT_MODEL ,
        messages=messages,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

In [21]:
print(ask("TAは何時にいる？"))

考えた量： 2848 文字
資料によると、TAは**平日の13時から18時**まで常駐しています。


In [17]:
print(ask("学生証忘れたらどうすればいい？"))
print(ask("カラー印刷できる？"))
print(ask("演習室でコーヒー飲んでいい？"))
print(ask("演習室のWi-Fiのパスワードは？"))   # ← 資料にない質問

資料によると、学生証を忘れた場合は**事務室で当日限りの仮カードを借りられます**。ただし、仮カードは17時までに返却する必要があります。
カラー印刷はできません。
資料によると、「演習室内での飲食は禁止」ですが、「ふた付きの飲み物は持ち込めます」と記載されています。したがって、コーヒーもふた付きであれば持ち込み可能です。
資料に記載にありません。


# まとめ
1. 資料を章ごとに切る(chuck_by_paragraph)
2. 全部の塊を数字にする(embed)
3. 質問に近い塊を探す(search + cos_sim)
4. messageに詰めてAIに渡す(ask)

In [ ]:
### 高速化 ### 
def ask(question):
    results = search(question, k=2)
    context = "\n---\n".join(chunk for score, chunk in results)

    messages = [
        {"role": "system", "content": "資料だけを根拠に答えて。資料にないことは「資料に記載にありません」と答えて。"},
        {"role": "user", "content": f"資料:\n{context}\n\n質問: {question}"},
    ]

    response = client.chat.completions.create(
        model=CHAT_MODEL ,
        messages=messages,
        temperature=0.2,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print("考えた量：", len(getattr(response.choices[0].message, "reasoning_content", "") or ""), "文字")
    return response.choices[0].message.content.strip()
# 自分のパソコン環境では効かなかった